In [3]:
from SPDE_problems import Data_to_solver
import torch
from dolfinx import mesh, fem
import ufl
import numpy as np
from FEniCSx_solver import interpolate_expr


for idx in range(88):

     fs , G = Data_to_solver(idx, train = True)

     cfun = fem.Function(fs.Yh)
     ptx = torch.tensor(fs.Yh.tabulate_dof_coordinates()[:,0], dtype=torch.float32).view(-1,1)
     pty = torch.tensor(fs.Yh.tabulate_dof_coordinates()[:,1], dtype=torch.float32).view(-1,1)

     x = G.x
     Duh_norm = torch.sqrt(x[:,8]**2 + x[:,7]**2).view(-1,1)
     uh_max = torch.tensor(np.max(fs.uh.x.array[fs.domain.geometry.dofmap], axis=1), dtype=torch.float32).view(-1,1)
     uh_min = torch.tensor(np.min(fs.uh.x.array[fs.domain.geometry.dofmap], axis=1), dtype=torch.float32).view(-1,1)

     Jinv = ufl.JacobianInverse(fs.domain)


     b_norm = ufl.sqrt(ufl.dot(fs.b, fs.b))

     h_K = 2 * b_norm / ufl.sqrt(ufl.dot(fs.b, Jinv.T * Jinv * fs.b))
     #h_K = ufl.CellDiameter(domain=fs.domain) 
     cfun = interpolate_expr(h_K, fs.Yh)
     h = torch.tensor(cfun.x.array.copy(), dtype=torch.float32)
     alpha = b_norm*h_K/(2*fs.eps)
     Xi = (1/ufl.tanh(alpha)-1/alpha)
     tau_K = h_K/(2*b_norm)*Xi
     cfun = interpolate_expr(tau_K, fs.Yh)
     tau_K = torch.tensor(cfun.x.array.copy(), dtype=torch.float32).view(-1,1)

     residual = ufl.dot(fs.b, ufl.grad(fs.uh))

     if fs.c != None:
          residual +=fs.c*fs.uh

     cfun = interpolate_expr(residual, fs.Yh)
     residual = torch.tensor(cfun.x.array.copy(), dtype=torch.float32)
     Pe = (torch.sqrt(x[:,1]**2+x[:,2]**2)*h/(2*x[:,0])).view(-1,1)

     Jdet = ufl.JacobianDeterminant(fs.domain)
     cfun = interpolate_expr(Jdet, fs.Yh)
     Jdet = torch.tensor(cfun.x.array.copy(), dtype=torch.float32).view(-1,1)
     c = x[:,3].view(-1,1)
     f = x[:,4].view(-1,1)
     b1 = x[:,1]
     b2 = x[:,2]
     edge_attr = []
     for e in range(len(G.edge_index[0])):
          ids, idt = G.edge_index[0,e], G.edge_index[1,e]
          cd = x[ids,5]
          dx = ptx[idt] - ptx[ids]
          dy = pty[idt] - pty[ids]
          dist = torch.sqrt(dx**2+dy**2)
          if dist > 1e-16:
               sprod = ((dx*b1[ids] + dx*b2[ids])/torch.sqrt(b1[ids]**2 + b2[ids]**2)  + x[ids,0])/dist
          else:
               sprod = c[ids]

          edge_attr.append([sprod, (1/cd)**(-dist)])
     
     edge_attr = torch.tensor(edge_attr, dtype=torch.float32)

         
     G.x = torch.cat([tau_K, f, ptx, pty, Duh_norm, uh_min, uh_max],dim=1)
     G.edge_attr = edge_attr

     torch.save(G, f"data/training_set_v2/input_values/raw/G_{idx}.pt")
     print(idx, G)

0 Data(x=[64, 7], edge_index=[2, 484], y=[64, 1], prblm_id=0, mesh_id=0, upper=[64, 1], edge_attr=[484, 2])
1 Data(x=[128, 7], edge_index=[2, 1012], y=[128, 1], prblm_id=0, mesh_id=1, upper=[128, 1], edge_attr=[1012, 2])
2 Data(x=[256, 7], edge_index=[2, 2068], y=[256, 1], prblm_id=0, mesh_id=2, upper=[256, 1], edge_attr=[2068, 2])
3 Data(x=[512, 7], edge_index=[2, 4180], y=[512, 1], prblm_id=0, mesh_id=3, upper=[512, 1], edge_attr=[4180, 2])
4 Data(x=[128, 7], edge_index=[2, 1012], y=[128, 1], prblm_id=0, mesh_id=4, upper=[128, 1], edge_attr=[1012, 2])
5 Data(x=[256, 7], edge_index=[2, 2116], y=[256, 1], prblm_id=0, mesh_id=5, upper=[256, 1], edge_attr=[2116, 2])
6 Data(x=[512, 7], edge_index=[2, 4324], y=[512, 1], prblm_id=0, mesh_id=6, upper=[512, 1], edge_attr=[4324, 2])
7 Data(x=[1024, 7], edge_index=[2, 8740], y=[1024, 1], prblm_id=0, mesh_id=7, upper=[1024, 1], edge_attr=[8740, 2])
8 Data(x=[256, 7], edge_index=[2, 2068], y=[256, 1], prblm_id=0, mesh_id=8, upper=[256, 1], edge_a

In [2]:
from SPDE_problems import Data_to_solver
import torch
from dolfinx import mesh, fem
import ufl
import numpy as np
from FEniCSx_solver import interpolate_expr

for idx in range(15):

    fs , G = Data_to_solver(idx, train = False)

    cfun = fem.Function(fs.Yh)
    ptx = torch.tensor(fs.Yh.tabulate_dof_coordinates()[:,0], dtype=torch.float32).view(-1,1)
    pty = torch.tensor(fs.Yh.tabulate_dof_coordinates()[:,1], dtype=torch.float32).view(-1,1)

    x = G.x
    Duh_norm = torch.sqrt(x[:,8]**2 + x[:,7]**2).view(-1,1)
    uh_max = torch.tensor(np.max(fs.uh.x.array[fs.domain.geometry.dofmap], axis=1), dtype=torch.float32).view(-1,1)
    uh_min = torch.tensor(np.min(fs.uh.x.array[fs.domain.geometry.dofmap], axis=1), dtype=torch.float32).view(-1,1)

    Jinv = ufl.JacobianInverse(fs.domain)


    b_norm = ufl.sqrt(ufl.dot(fs.b, fs.b))

    h_K = 2 * b_norm / ufl.sqrt(ufl.dot(fs.b, Jinv.T * Jinv * fs.b))
    #h_K = ufl.CellDiameter(domain=fs.domain) 
    cfun = interpolate_expr(h_K, fs.Yh)
    h = torch.tensor(cfun.x.array.copy(), dtype=torch.float32)
    alpha = b_norm*h_K/(2*fs.eps)
    Xi = (1/ufl.tanh(alpha)-1/alpha)
    tau_K = h_K/(2*b_norm)*Xi
    cfun = interpolate_expr(tau_K, fs.Yh)
    tau_K = torch.tensor(cfun.x.array.copy(), dtype=torch.float32).view(-1,1)

    residual = ufl.dot(fs.b, ufl.grad(fs.uh))

    if fs.c != None:
        residual +=fs.c*fs.uh

    cfun = interpolate_expr(residual, fs.Yh)
    residual = torch.tensor(cfun.x.array.copy(), dtype=torch.float32)
    Pe = (torch.sqrt(x[:,1]**2+x[:,2]**2)*h/(2*x[:,0])).view(-1,1)

    Jdet = ufl.JacobianDeterminant(fs.domain)
    cfun = interpolate_expr(Jdet, fs.Yh)
    Jdet = torch.tensor(cfun.x.array.copy(), dtype=torch.float32).view(-1,1)
    c = x[:,3].view(-1,1)
    f = x[:,4].view(-1,1)
    b1 = x[:,1]
    b2 = x[:,2]
    edge_attr = []
    for e in range(len(G.edge_index[0])):
        ids, idt = G.edge_index[0,e], G.edge_index[1,e]
        cd = x[ids,5]
        dx = ptx[idt] - ptx[ids]
        dy = pty[idt] - pty[ids]
        dist = torch.sqrt(dx**2+dy**2)
        if dist > 1e-16:
            sprod = ((dx*b1[ids] + dx*b2[ids])/torch.sqrt(b1[ids]**2 + b2[ids]**2)  + x[ids,0])/dist
        else:
            sprod = c[ids]

        edge_attr.append([sprod, (1/cd)**(-dist)])
    
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32)

        
    G.x = torch.cat([tau_K, f, ptx, pty, Duh_norm, uh_min, uh_max],dim=1)
    G.edge_attr = edge_attr

    torch.save(G, f"data/test_set_v2/input_values/raw/G_{idx}.pt")
    print(idx, G)

0 Data(x=[1054, 7], edge_index=[2, 9100], y=[1054, 1], prblm_id=0, mesh_id=0, upper=[1054, 1], edge_attr=[9100, 2])
1 Data(x=[1860, 7], edge_index=[2, 23214], y=[1860, 1], prblm_id=0, mesh_id=1, upper=[1860, 1], edge_attr=[23214, 2])
2 Data(x=[1320, 7], edge_index=[2, 11446], y=[1320, 1], prblm_id=1, mesh_id=2, upper=[1320, 1], edge_attr=[11446, 2])
3 Data(x=[3280, 7], edge_index=[2, 41354], y=[3280, 1], prblm_id=1, mesh_id=3, upper=[3280, 1], edge_attr=[41354, 2])
4 Data(x=[1147, 7], edge_index=[2, 9919], y=[1147, 1], prblm_id=2, mesh_id=4, upper=[1147, 1], edge_attr=[9919, 2])
5 Data(x=[2310, 7], edge_index=[2, 28952], y=[2310, 1], prblm_id=2, mesh_id=5, upper=[2310, 1], edge_attr=[28952, 2])
6 Data(x=[1155, 7], edge_index=[2, 9991], y=[1155, 1], prblm_id=3, mesh_id=6, upper=[1155, 1], edge_attr=[9991, 2])
7 Data(x=[2574, 7], edge_index=[2, 32320], y=[2574, 1], prblm_id=3, mesh_id=7, upper=[2574, 1], edge_attr=[32320, 2])
8 Data(x=[1353, 7], edge_index=[2, 11737], y=[1353, 1], prblm_